In [1]:
import os
import torch
import pandas as pd

from transformers import (
    AutoProcessor,
    AutoModelForImageClassification,
    BlipProcessor,
    BlipForConditionalGeneration
)

from PIL import Image
import matplotlib.pyplot as plt
import evaluate
from nltk.translate.bleu_score import sentence_bleu

2025-09-19 09:03:15.054450: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv("../results/colonoscopy_metadata.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

Dataset shape: (42126, 7)
Columns: ['source', 'question', 'answer', 'img_id', 'question_type', 'question_norm', 'answer_type']
               source                                           question  \
0  Ulcerative Colitis  Are there any abnormalities in the image? Chec...   
1  Ulcerative Colitis  Are there any anatomical landmarks in the imag...   
2  Ulcerative Colitis  Are there any instruments in the image? Check ...   

               answer                     img_id question_type  \
0  ulcerative colitis  cla820gl0s3nv071u4fgd7xgq        Yes/No   
1                none  cla820gl0s3nv071u4fgd7xgq        Yes/No   
2                none  cla820gl0s3nv071u4fgd7xgq        Yes/No   

                                       question_norm answer_type  
0  are there any abnormalities in the image? chec...       Token  
1  are there any anatomical landmarks in the imag...     None/NA  
2  are there any instruments in the image? check ...     None/NA  


In [3]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Example test
preds = ["the mucosa looks inflamed"]
refs = [["the mucosa is inflamed"]]

print("BLEU:", bleu.compute(predictions=preds, references=refs))
print("ROUGE:", rouge.compute(predictions=preds, references=refs))

BLEU: {'bleu': 0.0, 'precisions': [0.75, 0.3333333333333333, 0.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 4, 'reference_length': 4}
ROUGE: {'rouge1': np.float64(0.75), 'rouge2': np.float64(0.3333333333333333), 'rougeL': np.float64(0.75), 'rougeLsum': np.float64(0.75)}


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-vqa-base").to(device)
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")

Using device: cuda


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
# Path where images are stored
IMG_DIR = "../images"

# Take one example row
sample = df.iloc[0]
image_path = os.path.join(IMG_DIR, sample["img_id"] + ".jpg")

if os.path.exists(image_path):
    image = Image.open(image_path).convert("RGB")
    question = sample["question"]

    inputs = blip_processor(image, question, return_tensors="pt").to(device)
    out = blip_model.generate(**inputs)
    prediction = blip_processor.decode(out[0], skip_special_tokens=True)

    print("Question:", question)
    print("Ground Truth Answer:", sample["answer"])
    print("BLIP Prediction:", prediction)
else:
    print("⚠️ Image not found:", image_path)

Question: Are there any abnormalities in the image? Check all that are present.
Ground Truth Answer: ulcerative colitis
BLIP Prediction: are there any abnormalities in the image? check all that are present. 5


In [6]:
subset = df.sample(20, random_state=42)

preds, refs = [], []

for _, row in subset.iterrows():
    image_path = os.path.join(IMG_DIR, row["img_id"] + ".jpg")
    if not os.path.exists(image_path):
        continue
    
    image = Image.open(image_path).convert("RGB")
    question = row["question"]
    answer = row["answer"]

    inputs = blip_processor(image, question, return_tensors="pt").to(device)
    out = blip_model.generate(**inputs)
    prediction = blip_processor.decode(out[0], skip_special_tokens=True)

    preds.append(prediction)
    refs.append([answer])  # wrapped in list for evaluate

# Compute metrics
print("Subset size:", len(preds))
print("BLEU:", bleu.compute(predictions=preds, references=refs))
print("ROUGE:", rouge.compute(predictions=preds, references=refs))

Subset size: 20
BLEU: {'bleu': 0.0, 'precisions': [0.0, 0.0, 0.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 8.333333333333334, 'translation_length': 175, 'reference_length': 21}
ROUGE: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
